# Evaluate CNN3DMultiAtt

Load fold checkpoints, compute classification metrics and display confusion matrices.

In [ ]:
import os
import sys
from pathlib import Path

ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import matplotlib.pyplot as plt
import numpy as np
import torch
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)

import config
from data import experiment_name, get_fold_loaders, set_seed
from model import CNN3DMultiAtt

set_seed(config.DEFAULTS["seed"])

GPU = "0"
if GPU is not None:
    os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
    os.environ["CUDA_VISIBLE_DEVICES"] = GPU

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

## Match the training configuration

Use the same hyperparameters as the training run, or set `run_name` to an existing output directory.

In [ ]:
HP = dict(
    lr=1e-3,
    epochs=15,
    batch_size=32,
    num_classes=2,
    dropout_p=0.0,
    spd_depth=1,
    reduction=1,
    att_position=[1, 0, 1, 0, 1, 0],
    n_folds=5,
    suffix="_v1",
    # Optionally override the experiment folder name:
    # run_name="lr0.001_epochs15_bz32_v1_red1_spd1",
    run_name=None,
)

name = HP["run_name"] or experiment_name(
    lr=HP["lr"],
    epochs=HP["epochs"],
    batch_size=HP["batch_size"],
    reduction=HP["reduction"],
    spd_depth=HP["spd_depth"],
    suffix=HP["suffix"],
)
weights_path = config.OUTPUT_DIR / CNN3DMultiAtt.model_name / name / "weights"
print("Evaluating run:", name)
print("Weights path:", weights_path)

## Evaluation

In [ ]:
def fpr_at_tpr95(labels, probs):
    fpr, tpr, _ = roc_curve(labels, probs)
    idx = np.where(tpr >= 0.95)[0]
    return float(fpr[idx[0]]) if len(idx) > 0 else float("nan")


def evaluate_fold(model, loaders, device):
    all_preds, all_labels, all_probs = [], [], []
    with torch.no_grad():
        for (lb0, ll0), (lb1, _), (lb2, _) in zip(*loaders):
            logits = model([lb0.to(device), lb1.to(device), lb2.to(device)])
            probs = torch.softmax(logits, dim=1)[:, 1]
            preds = torch.argmax(logits, dim=1)
            all_probs.extend(probs.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(ll0.cpu().numpy())

    metrics = {
        "accuracy": accuracy_score(all_labels, all_preds),
        "precision": precision_score(all_labels, all_preds, average="binary", zero_division=0),
        "recall": recall_score(all_labels, all_preds, average="binary", zero_division=0),
        "f1": f1_score(all_labels, all_preds, average="binary", zero_division=0),
    }

    if len(np.unique(all_labels)) < 2:
        metrics.update({"auc_roc": np.nan, "auc_pr": np.nan, "fpr95": np.nan, "specificity": np.nan})
    else:
        tn, fp, fn, tp = confusion_matrix(all_labels, all_preds).ravel()
        metrics["auc_roc"] = roc_auc_score(all_labels, all_probs)
        metrics["auc_pr"] = average_precision_score(all_labels, all_probs)
        metrics["fpr95"] = fpr_at_tpr95(all_labels, all_probs)
        metrics["specificity"] = tn / (tn + fp) if (tn + fp) > 0 else np.nan

    return metrics, confusion_matrix(all_labels, all_preds)


metrics_by_fold = {
    k: [] for k in ("accuracy", "precision", "recall", "f1", "auc_roc", "auc_pr", "fpr95", "specificity")
}
cm_list = []

for fold in range(HP["n_folds"]):
    print(f"\n--- Fold {fold} ---")
    model = CNN3DMultiAtt(
        num_classes=HP["num_classes"],
        dropout_p=HP["dropout_p"],
        spd_depth=HP["spd_depth"],
        reduction=HP["reduction"],
        att_position=HP["att_position"],
    ).to(device)

    ckpt = weights_path / f"fold_{fold}.pth"
    model.load_state_dict(torch.load(ckpt, map_location=device, weights_only=True))
    model.eval()

    data = get_fold_loaders(
        fold=fold,
        path_data=config.PATH_DATA,
        path_json_info=config.PATH_JSON_INFO,
        path_folds=config.PATH_FOLDS,
        batch_size=HP["batch_size"],
        modality=(1, 1, 1),
    )
    loaders = [data["T2"]["test_loader"], data["ADC"]["test_loader"], data["HBV"]["test_loader"]]

    metrics, cm = evaluate_fold(model, loaders, device)
    for k, v in metrics.items():
        metrics_by_fold[k].append(v)
    cm_list.append(cm)

    print(
        f"Acc={metrics['accuracy']:.4f}  Prec={metrics['precision']:.4f}  "
        f"Rec={metrics['recall']:.4f}  F1={metrics['f1']:.4f}  "
        f"AUC={metrics['auc_roc']:.4f}  Spec={metrics['specificity']:.4f}"
    )

print("\nMean ± Std over folds:")
for key, values in metrics_by_fold.items():
    arr = np.array(values, dtype=float)
    print(f"  {key:12s}  {np.nanmean(arr) * 100:6.2f} ± {np.nanstd(arr) * 100:5.2f}")

n = len(cm_list)
fig, axes = plt.subplots(1, n, figsize=(4 * n, 4))
if n == 1:
    axes = [axes]
for i, ax in enumerate(axes):
    ConfusionMatrixDisplay(confusion_matrix=cm_list[i], display_labels=[0, 1]).plot(
        ax=ax, cmap=plt.cm.Blues, colorbar=False
    )
    ax.set_title(f"Fold {i}")
plt.tight_layout()
plt.show()

## Forward-pass sanity check

Verify tensor shapes with a synthetic mini-batch.

In [ ]:
m = CNN3DMultiAtt(
    spd_depth=HP["spd_depth"],
    reduction=HP["reduction"],
    att_position=HP["att_position"],
).to(device)

dummy = [
    torch.randn(2, 1, 8, 64, 64, device=device),
    torch.randn(2, 1, 8, 64, 64, device=device),
    torch.randn(2, 1, 8, 64, 64, device=device),
]
with torch.no_grad():
    out = m(dummy)
print("logits:", tuple(out.shape))  # expected (2, 2)